# PyVISA - talking to the simulated oscilloscope

This is the simulator version of `PyVISA.ipynb`.  The instrument is
`virtual_scope.py`, a virtual 2-channel oscilloscope that speaks SCPI over a TCP
socket.  Start it in a terminal first:

    uv run virtual_scope.py

The point of the exercise is that **the client code barely changes**.  Only
the resource string is different - the SCPI commands are the same ones you
would send to the real instrument over USB.

Three things do need care with a raw-socket instrument, and each is marked
`# SIM:` in the cells below:

1. `rm.list_resources()` cannot discover socket instruments - type the
   address instead.
2. Termination characters must be set explicitly, or every query hangs.
3. The simulator implements a subset of the instrument's command set.  Sending
   a query it does not know gets you no reply at all, which shows up as
   `VI_ERROR_TMO` after the timeout expires.

---

In [ ]:
# Initialization
from warnings import filterwarnings
filterwarnings('ignore')

# Plot in the same cell
%matplotlib inline

### List resources, open resource

In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
print(rm)                       # which backend: pyvisa-py or NI-VISA

# SIM: list_resources() finds USB/GPIB/serial instruments by enumerating the
# bus.  A raw socket has no such enumeration, so the simulator will NOT
# appear here - the tuple is usually empty on a PC with no other instruments.
print("discovered:", rm.list_resources())

# SIM: so we name the resource explicitly instead of taking list_resources()[0].
#      For the real scope over USB this would be something like
#      "USB0::0x2A8D::0x0396::CN12345678::INSTR"
RESOURCE = "TCPIP0::127.0.0.1::5025::SOCKET"

mso = rm.open_resource(RESOURCE, open_timeout=5000)

# SIM: a socket instrument carries no terminator information, so PyVISA has to
#      be told.  Without this every query blocks until it times out.
mso.read_termination = "\n"
mso.write_termination = "\n"
mso.timeout = 5000              # ms

print(mso.query("*IDN?").strip())

### A habit worth forming: check the error queue

An instrument almost never refuses a command out loud.  A malformed command
is dropped and an entry is pushed onto its error queue, so a script can look
perfectly healthy while doing nothing at all.  Call this after anything you
are unsure about.

In [ ]:
def check_errors(instr=None):
    """Drain the instrument's error queue.  Returns a list of messages."""
    instr = instr or mso
    found = []
    while True:
        err = instr.query(":SYSTem:ERRor?").strip()
        if err.startswith("+0"):
            break
        found.append(err)
        if len(found) > 20:
            break
    return found

print("errors so far:", check_errors() or "none")

### Change settings

In [ ]:
# Autoscale: the simulator picks sensible V/div, timebase and trigger level
# for whatever signal is on channel 1, exactly as the real instrument does.
mso.write(":AUToscale")
print("after autoscale:",
      mso.query(":CHANnel1:SCALe?").strip(), "V/div,",
      mso.query(":TIMebase:SCALe?").strip(), "s/div")

In [ ]:
# SIM: ":TRIGger:MODE EDGE" is NOT implemented by virtual_scope.py.  The write is
#      discarded with -113,"Undefined header: MODE" and the matching *query*
#      never answers, so mso.query(":TRIGger:MODE?") raises VI_ERROR_TMO.
#      The simulator only ever does edge triggering, so there is nothing to set.
#
# mso.write(":TRIGger:MODE EDGE")            # <- would be ignored
# print(mso.query(":TRIGger:MODE?"))         # <- would time out

# The edge parameters themselves are supported:
mso.write(":TRIGger:EDGE:SOURce CHANnel1")
print(f'Trigger edge source: {mso.query(":TRIGger:EDGE:SOURce?").strip()}')

mso.write(":TRIGger:EDGE:LEVel 0.0")
print(f'Trigger edge level : {mso.query(":TRIGger:EDGE:LEVel?").strip()}')

mso.write(":TRIGger:EDGE:SLOPe POSitive")
print(f'Trigger edge slope : {mso.query(":TRIGger:EDGE:SLOPe?").strip()}')

print("errors:", check_errors() or "none")

In [ ]:
# Set vertical scale and offset.
#
# SIM: the real instrument accepts a suffixed argument like "500mv".  The
#      simulator parses arguments with float(), so "500mv" is rejected with
#      -100,"Command error".  Send a plain number in volts - which the real
#      instrument accepts too, so this form works on both.
mso.write(":CHANnel1:SCALe 1.0")
print(f'Channel 1 vertical scale: {mso.query(":CHANnel1:SCALe?").strip()}')

mso.write(":CHANnel1:OFFSet 0.0")           # volts, no "V" suffix
print(f'Channel 1 offset        : {mso.query(":CHANnel1:OFFSet?").strip()}')

print("errors:", check_errors() or "none")

In [ ]:
# Set horizontal scale and position.  Both are accepted as written.
mso.write(":TIMebase:SCALe 100e-6")
print(f'Timebase scale   : {mso.query(":TIMebase:SCALe?").strip()}')

mso.write(":TIMebase:POSition 0.0")
print(f'Timebase position: {mso.query(":TIMebase:POSition?").strip()}')

print("errors:", check_errors() or "none")

Note that the simulator *snaps* the scale settings to the 1-2-5 sequence the
real front-panel knobs use.  Ask for 0.3 V/div and you get 0.2 - the nearest
position on a logarithmic measure - just as you would on the bench.  Always
read the value back rather than assuming the setting took the value you sent.

In [ ]:
for asked in (0.3, 0.03, 1.7):
    mso.write(f":CHANnel1:SCALe {asked}")
    print(f"asked {asked:>5} V/div  ->  got {float(mso.query(':CHANnel1:SCALe?')):G} V/div")

mso.write(":CHANnel1:SCALe 0.5")            # put it back

### Make measurements

In [ ]:
# A frequency measurement needs at least two cycles on screen, so widen the
# timebase first: at 100 us/div a 1 kHz sine gives exactly one cycle across
# the 10 divisions and the instrument reports 9.9E37, "cannot measure".
mso.write(":TIMebase:SCALe 500e-6")         # 5 ms across the screen

# :DIGitize takes one acquisition and holds it.  Do this before reading a
# preamble and its data, otherwise each query lands on a different sweep -
# true of the real instrument as well as the simulator.
mso.write(":DIGitize CHANnel1")

# SIM: ":MEASure:SOURce" is not implemented.  Worse, a bare
#      mso.write(":MEASure:FREQuency") makes the simulator *reply* even
#      though no "?" was sent, leaving an unread line in the socket that
#      desynchronises every later query.  Name the channel in the query
#      instead - the documented form, and unambiguous.
freq = float(mso.query(":MEASure:FREQuency? CHANnel1"))
vamp = float(mso.query(":MEASure:VAMPlitude? CHANnel1"))
vpp  = float(mso.query(":MEASure:VPP? CHANnel1"))
vrms = float(mso.query(":MEASure:VRMS? CHANnel1"))

NOT_AVAILABLE = 9.9e37          # the "could not measure" value
def show(name, value, unit):
    print(f"{name:>28s}: " +
          ("not available (signal clipped or <2 cycles on screen)"
           if value >= NOT_AVAILABLE else f"{value:.6G} {unit}"))

show("Frequency on channel 1", freq, "Hz")
show("Vertical amplitude", vamp, "V")
show("Peak to peak", vpp, "V")
show("RMS", vrms, "V")

mso.write(":RUN")               # release the hold

### Download waveform data

This is where a socket instrument behaves exactly like the real one: the
data comes back as an IEEE 488.2 definite-length block, and the scaling
factors needed to turn the raw codes into volts and seconds come from the
preamble.

In [ ]:
mso.write(":WAVeform:SOURce CHANnel1")
print(f'Waveform source: {mso.query(":WAVeform:SOURce?").strip()}')

mso.write(":WAVeform:FORMat BYTE")          # 1 byte per point, 0..255
print(f'Waveform format: {mso.query(":WAVeform:FORMat?").strip()}')

# SIM: ":WAVeform:POINts:MODE RAW" is not implemented - the write is ignored
#      and the query would time out.  The simulator always returns the whole
#      record, which is what RAW means anyway.
#
# mso.write(":WAVeform:POINts:MODE RAW")    # <- would be ignored

mso.write(":WAVeform:POINts 10000")
print(f'Waveform points: {mso.query(":WAVeform:POINts?").strip()}')

In [ ]:
# SIM: the individual :WAVeform:XINCrement? / :YREFerence? / ... queries are
#      not implemented and would each time out.  Use :WAVeform:PREamble?,
#      which returns all ten numbers in one go and works on both instruments.
mso.write(":DIGitize CHANnel1")             # freeze one acquisition

pre = mso.query(":WAVeform:PREamble?").strip().split(",")
(wav_format, wav_type, points, count,
 x_increment, x_origin, x_reference,
 y_increment, y_origin, y_reference) = [float(p) for p in pre]

print(f"points      : {int(points)}")
print(f"x_increment : {x_increment:.6E} s")
print(f"x_origin    : {x_origin:.6E} s")
print(f"y_increment : {y_increment:.6E} V")
print(f"y_origin    : {y_origin:.6E} V")
print(f"y_reference : {y_reference:G}")

In [ ]:
# datatype="B" is one unsigned byte per point, matching :WAVeform:FORMat BYTE.
raw = mso.query_binary_values(":WAVeform:DATA?", datatype="B")
print(f"Number of data values: {len(raw)}")

times = [x_origin + (i - x_reference) * x_increment for i in range(len(raw))]
volts = [(code - y_reference) * y_increment + y_origin for code in raw]

print(f"first point: t={times[0]:.6E} s  v={volts[0]:+.4f} V")
print(f"last  point: t={times[-1]:.6E} s  v={volts[-1]:+.4f} V")

mso.write(":RUN")

In [ ]:
import os

# Write to a new file so the waveform_data.csv shipped with the repository - the
# one Exercise 7 works from - is left untouched.
os.makedirs("data", exist_ok=True)          # the original notebook assumed this existed
with open("data/waveform_data_new.csv", "w") as f:
    f.write("time_s,voltage_V\n")
    for t, v in zip(times, volts):
        f.write(f"{t},{v}\n")

print(f"Wrote {len(times)} points to data/waveform_data_new.csv")

### The screen

SIM: a real scope can hand you a PNG of its own display with
`:DISPlay:DATA? PNG, COLor`.  `virtual_scope.py` has no framebuffer, so that
query would time out.  Instead we redraw the screen ourselves from the data
we just downloaded - which is also a useful exercise on real hardware, since
you get a plot you can style and annotate rather than a screenshot.

In [ ]:
import matplotlib.pyplot as plt

v_div = float(mso.query(":CHANnel1:SCALe?"))
t_div = float(mso.query(":TIMebase:SCALe?"))
offset = float(mso.query(":CHANnel1:OFFSet?"))
position = float(mso.query(":TIMebase:POSition?"))

H_DIV, V_DIV = 10, 8            # the scope screen is 10 x 8 divisions

fig, ax = plt.subplots(figsize=(9, 5.4), facecolor="#0d1117")
ax.set_facecolor("#0d1117")
ax.plot([t * 1e3 for t in times], volts, color="#ffd400", linewidth=1.2)

ax.set_xlim((position - H_DIV * t_div / 2) * 1e3,
            (position + H_DIV * t_div / 2) * 1e3)
ax.set_ylim(offset - V_DIV * v_div / 2, offset + V_DIV * v_div / 2)
ax.set_xticks([(position + i * t_div) * 1e3 for i in range(-H_DIV // 2, H_DIV // 2 + 1)])
ax.set_yticks([offset + i * v_div for i in range(-V_DIV // 2, V_DIV // 2 + 1)])
ax.grid(True, color="#2a3038", linewidth=0.8)
ax.tick_params(colors="#8b949e", labelsize=8)
for s in ax.spines.values():
    s.set_color("#30363d")

ax.set_xlabel("time (ms)", color="#8b949e")
ax.set_ylabel("volts", color="#8b949e")
ax.set_title(f"CH1   {v_div:G} V/div   {t_div * 1e6:G} us/div",
             color="#c9d1d9", fontsize=10)
plt.tight_layout()
plt.show()

### Reading back what the GUI set

This is the part worth demonstrating in class.  `virtual_scope.py` is just
another VISA client, and the instrument holds a single set of settings.  So
whatever you turn on the front panel, this notebook can read - and the other
way round.

Start the GUI in a third terminal while the simulator and this notebook are
running:

    uv run virtual_scope.py

Press **Connect**, move the **Volts / div** and **Time / div** controls, then
run the cell below.  It reports what the instrument is actually set to, which
is what the GUI just wrote.

In [ ]:
def read_front_panel(channel=1):
    """Everything scope_gui.py can set, read straight back off the instrument."""
    return {
        "channel":   channel,
        "v_div":     float(mso.query(f":CHANnel{channel}:SCALe?")),
        "offset":    float(mso.query(f":CHANnel{channel}:OFFSet?")),
        "t_div":     float(mso.query(":TIMebase:SCALe?")),
        "position":  float(mso.query(":TIMebase:POSition?")),
        "trigger":   float(mso.query(":TRIGger:EDGE:LEVel?")),
        "coupling":  mso.query(f":CHANnel{channel}:COUPling?").strip(),
    }

state = read_front_panel(1)
for k, v in state.items():
    print(f"  {k:9s}: {v}")

In [ ]:
# Run this cell repeatedly while moving the sliders in virtual_scope.py - each
# run shows the instrument's current state.  A real bench scope behaves the
# same way when a colleague turns a knob while your script is running.
import time

for _ in range(5):
    s = read_front_panel(1)
    print(f"{s['v_div']:>8G} V/div   {s['t_div']:>10G} s/div   "
          f"offset {s['offset']:+.2f} V   trigger {s['trigger']:+.2f} V")
    time.sleep(1.0)

It works in the other direction too.  Run the next cell and watch
`virtual_scope.py` follow: its poll loop re-reads the instrument every 250 ms, so
its dropdowns and sliders will move to match what the notebook just wrote.

In [ ]:
mso.write(":CHANnel1:SCALe 0.2")            # 200 mV/div
mso.write(":CHANnel1:OFFSet 1.0")           # 1 V up
mso.write(":TIMebase:SCALe 500e-6")         # 500 us/div

print("notebook wrote:", read_front_panel(1))
print("errors:", check_errors() or "none")
print("\n-> the virtual_scope.py window should now show 200 mV/div and 500 us/div")

### Close down

In [ ]:
mso.write(":RUN")               # leave the instrument acquiring
mso.close()

In [ ]:
rm.close()
print("session closed")